# Phase S1 v2: Cooling Theory Validation

**Goal**: Validate BatchNorm and Skip as "cooling mechanisms"

**Design**:
- 4 configs: None / LN / BN / Skip
- 2 D values: base_ch 32, 96
- 100 epochs per run
- **Checkpoint every 20 epochs**
- **Logging every 10 epochs with ETA**
- **CPU verification (1 epoch) before running**

**Expected time**: ~50 min on T4 GPU

## STEP 1: Clone Repo

In [ ]:
import os, subprocess, sys, json, math, time
from pathlib import Path

# Clone develop branch
if os.path.exists('ThermoRG-NN'):
    print("Removing existing ThermoRG-NN...", flush=True)
    subprocess.run(['rm', '-rf', 'ThermoRG-NN'], check=True)
    
print("Cloning ThermoRG-NN develop...", flush=True)
subprocess.run(['git', 'clone', '-b', 'develop', 'https://github.com/xliu203/ThermoRG-NN.git'], check=True)
sys.path.insert(0, 'ThermoRG-NN')
os.chdir('ThermoRG-NN')

print(f"CWD: {os.getcwd()}", flush=True)
print("Clone done!", flush=True)

## STEP 2: CPU Verification (1 epoch, 1 batch)

In [ ]:
# CPU verification - test that everything works before GPU run
import torch
import torch.nn as nn
import numpy as np

print("=== CPU VERIFICATION ===", flush=True)

# Test ThermoNet import
try:
    from thermorg.simulations.networks.thermonet import ThermoNet
    print("✅ ThermoNet import OK", flush=True)
except Exception as e:
    print(f"❌ ThermoNet import FAILED: {e}", flush=True)
    raise

# Test data loading
try:
    import torchvision.transforms as T
    from torchvision.datasets import CIFAR10
    from torch.utils.data import DataLoader
    
    transform = T.Compose([T.ToTensor()])
    ds = CIFAR10('./data', train=True, transform=transform, download=True)
    dl = DataLoader(ds, batch_size=32)
    print(f"✅ Data loading OK ({len(ds)} samples)", flush=True)
except Exception as e:
    print(f"❌ Data loading FAILED: {e}", flush=True)
    raise

# Test model creation
try:
    model = ThermoNet(in_channels=3, num_classes=10, base_channels=32, depth=3,
                      layer_norm=False, batch_norm=False, use_skip=True, dropout_rate=0.0, activation='gelu')
    print(f"✅ Model creation OK", flush=True)
except Exception as e:
    print(f"❌ Model creation FAILED: {e}", flush=True)
    raise

# Test 1 batch training on CPU
try:
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    X, y = next(iter(dl))
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()
    print(f"✅ 1-batch training OK (loss={loss.item():.4f})", flush=True)
except Exception as e:
    print(f"❌ Training FAILED: {e}", flush=True)
    raise

del model, optimizer, criterion, X, y
torch.cuda.empty_cache()
print("=== CPU VERIFICATION PASSED ===", flush=True)
print("Ready to run GPU experiments!", flush=True)

## STEP 3: GPU Config

In [ ]:
# Config
CONFIGS = [
    ('None_NoSkip',  'none',       False),
    ('LN_NoSkip',    'layernorm',  False),
    ('BN_NoSkip',    'batchnorm',  False),
    ('None_Skip',    'none',       True),
]

D_VALUES = [32, 96]  # base_ch
SEEDS = [42]
EPOCHS = 100
BATCH_SIZE = 128
LR = 0.01
MOMENTUM = 0.9
CHECKPOINT_EVERY = 20  # save checkpoint every N epochs
LOG_EVERY = 10        # log every N epochs

OUT_DIR = Path('./phase_s1_results_v2')
OUT_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}", flush=True)
if device.type != 'cuda':
    print("❌ WARNING: No GPU available!", flush=True)
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

print(f"\nConfigs: {len(CONFIGS)}", flush=True)
print(f"D values: {D_VALUES}", flush=True)
print(f"Epochs: {EPOCHS}", flush=True)
print(f"Total runs: {len(CONFIGS) * len(D_VALUES) * len(SEEDS)}", flush=True)
print(f"\nEstimated time: ~50 min on T4 GPU", flush=True)

## STEP 4: Data Loading

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
transform_val = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

train_ds = CIFAR10('./data', train=True, transform=transform_train, download=True)
val_ds   = CIFAR10('./data', train=False, transform=transform_val, download=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}", flush=True)
print(f"Batches per epoch: {len(train_dl)}", flush=True)

## STEP 5: Training Function with Checkpoint + Logging

In [ ]:
def train_one_model(model, epochs, lr, seed, checkpoint_path=None):
    """Train with checkpoint support and periodic logging."""
    start_epoch = 0
    best_loss = float('inf')
    
    # Resume from checkpoint if exists
    if checkpoint_path and checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt['epoch'] + 1
        best_loss = ckpt.get('best_loss', float('inf'))
        print(f"  ↪️  Resuming from epoch {start_epoch}, best_loss={best_loss:.4f}", flush=True)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    model = model.to(device)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    # Restore scheduler
    for _ in range(start_epoch):
        scheduler.step()
    
    t0 = time.time()
    
    for epoch in range(start_epoch, epochs):
        # Train
        model.train()
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
        # Evaluate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X, y in val_dl:
                X, y = X.to(device), y.to(device)
                val_loss += criterion(model(X), y).item() * X.size(0)
        val_loss /= len(val_ds)
        if val_loss < best_loss:
            best_loss = val_loss
        
        # Logging every N epochs
        if (epoch + 1) % LOG_EVERY == 0 or epoch == epochs - 1:
            elapsed = (time.time() - t0) / 60
            epochs_done = epoch - start_epoch + 1
            epochs_left = epochs - start_epoch - epochs_done
            eta = elapsed / epochs_done * epochs_left if epochs_done > 0 else 0
            print(f"  Epoch {epoch+1}/{epochs}: loss={val_loss:.4f}, best={best_loss:.4f}, elapsed={elapsed:.1f}min, ETA={eta:.1f}min", flush=True)
        
        # Save checkpoint
        if checkpoint_path and (epoch + 1) % CHECKPOINT_EVERY == 0:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_loss': best_loss,
            }, checkpoint_path)
            print(f"  💾 Checkpoint saved at epoch {epoch+1}", flush=True)
    
    return best_loss

print("Training function defined", flush=True)

## STEP 6: Main Loop

In [ ]:
from datetime import datetime
from thermorg.simulations.networks.thermonet import ThermoNet

print("\n" + "="*70, flush=True)
print("STARTING MAIN LOOP", flush=True)
print("="*70, flush=True)

results = {
    'timestamp': datetime.now().isoformat(),
    'epochs': EPOCHS,
    'total_runs': len(CONFIGS) * len(D_VALUES) * len(SEEDS),
    'configs': []
}

t_start = time.time()

for cfg_name, norm_type, use_skip in CONFIGS:
    print(f"\n{'='*60}", flush=True)
    print(f"CONFIG: [{cfg_name}] norm={norm_type}, skip={use_skip}", flush=True)
    print(f"{'='*60}", flush=True)
    
    cfg_result = {'name': cfg_name, 'norm': norm_type, 'skip': use_skip}
    cfg_start = time.time()
    
    for base_ch in D_VALUES:
        for seed in SEEDS:
            run_name = f"{cfg_name}_ch{base_ch}_s{seed}"
            ckpt_path = OUT_DIR / f"{run_name}.pt"
            
            print(f"\n--- {run_name} ---", flush=True)
            
            # Create model
            model = ThermoNet(
                in_channels=3, num_classes=10, base_channels=base_ch, depth=3,
                layer_norm=(norm_type == 'layernorm'),
                batch_norm=(norm_type == 'batchnorm'),
                use_skip=use_skip, dropout_rate=0.0, activation='gelu'
            )
            
            # Train
            t0 = time.time()
            best_loss = train_one_model(model, EPOCHS, LR, seed, ckpt_path)
            elapsed = (time.time() - t0) / 60
            
            print(f"  ✅ {run_name} DONE: best_loss={best_loss:.4f}, time={elapsed:.1f}min", flush=True)
            
            # Store
            key = f'ch{base_ch}'
            if key not in cfg_result:
                cfg_result[key] = {}
            cfg_result[key][f's{seed}'] = {'best_val_loss': best_loss, 'time_min': elapsed}
            
            del model
            torch.cuda.empty_cache()
    
    cfg_time = (time.time() - cfg_start) / 60
    cfg_result['total_time_min'] = cfg_time
    results['configs'].append(cfg_result)
    print(f"\n[{cfg_name}] total: {cfg_time:.1f} min", flush=True)

total_time = (time.time() - t_start) / 60
results['total_time_min'] = total_time
print(f"\n{'='*70}", flush=True)
print(f"ALL DONE! Total time: {total_time:.1f} min", flush=True)
print(f"{'='*70}", flush=True)

## STEP 7: Save & Summary

In [ ]:
# Save
out_file = OUT_DIR / 'phase_s1_results.json'
with open(out_file, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nSaved to {out_file}", flush=True)

# Summary
print("\n" + "="*60, flush=True)
print("SUMMARY", flush=True)
print("="*60, flush=True)
for cfg in results['configs']:
    losses = []
    for ch in D_VALUES:
        for s in SEEDS:
            if f'ch{ch}' in cfg and f's{s}' in cfg[f'ch{ch}']:
                losses.append(cfg[f'ch{ch}'][f's{s}']['best_val_loss'])
    avg = sum(losses)/len(losses) if losses else 0
    print(f"{cfg['name']:<15} avg_loss={avg:.4f}  time={cfg.get('total_time_min', 0):.1f}min", flush=True)
print("\n✅ DONE!", flush=True)